# 🧠 Long Short-Term Memory (LSTM) — The Complete Deep-Dive

**Topics covered:**
1. Why vanilla RNN fails (linguistic motivation + gradient proof)
2. LSTM architecture — Cell State vs Hidden State
3. All 4 gates with equations, intuition, and worked numbers
4. The Constant Error Carousel — why addition beats multiplication
5. How LSTM solves vanishing gradients (mathematical proof)
6. Step-by-step LSTM cell computation (numerical trace)
7. Bidirectional and Stacked LSTM
8. Peephole connections
9. Full end-to-end text classification training with loss curves
10. Visualisations for every concept
11. Senior-level interview Q&A


## 1. 🔥 Why Vanilla RNN Fails — The Memory Crisis

### The Linguistic Problem
Consider an RNN reading:
> "The **girl**, who was wearing a beautiful dress and standing near the crowded window, picked up **her** book."

To resolve the pronoun **"her"**, the model must remember the gender of the subject (**"girl"**) — 14 words earlier.

**What happens in a vanilla RNN?**
- At each step: $h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t)$
- After 14 steps with $W_{hh}=0.9$ and $\tanh' \approx 0.5$:
  $$(0.9 \times 0.5)^{14} \approx 0.45^{14} \approx 0.00007$$
- The gradient signal from "girl" to "her" is $\approx 7 \times 10^{-5}$ → effectively **zero**
- The weight update from this long-range dependency is **negligible** → model never learns it

### The Three Root Causes
| Problem | RNN Mechanism | Why It Hurts |
|---|---|---|
| No selectivity | $h_t$ always uses ALL of $h_{t-1}$ | Cannot keep "girl=female" silently for 14 steps |
| Multiplicative gradient | $(W_{hh})^T$ in BPTT | Small $W$ → vanishing; large $W$ → explosion |
| Overwrite problem | $h_t$ fully replaces $h_{t-1}$ | New input can erase important long-range context |

### LSTM's Solution in One Sentence
> LSTM adds a **separate "vault"** (Cell State) for long-term memory and **three programmable gates** to control what enters, leaves, or is erased from that vault.


## 2. 🏗️ LSTM Architecture — Two Highways

### Key Innovation: Two Separate State Vectors

| State | Symbol | Role | Lives in |
|---|---|---|---|
| **Cell State** | $C_t$ | Long-term memory vault | Flows mostly unchanged (additive) |
| **Hidden State** | $h_t$ | Short-term working memory / output | Updated every step via gates |

This separation is the **core innovation** of LSTM.

A vanilla RNN has ONE state ($h_t$) that must serve both roles simultaneously, causing "clutter" — it must remember "girl=female" from 14 steps ago AND decide the output word right now. These two jobs interfere.

LSTM **decouples** them:
- $C_t$ carries the "girl=female" fact silently across 14 steps (barely modified)
- $h_t$ focuses on the current word-to-word transition and output

### The Cell State Highway
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

Notice: this is **addition** ($+$), not multiplication of matrices.  
This is why gradients don't vanish — we'll prove this in Section 5.

### The Hidden State Path
$$h_t = o_t \odot \tanh(C_t)$$

The output gate filters which parts of the cell state are "published" as the visible output.


## 3. 🚪 The Four LSTM Equations — Intuition + Math

At each time step, LSTM receives:
- $x_t$     → current input
- $h_{t-1}$ → previous hidden state (short-term memory)
- $C_{t-1}$ → previous cell state (long-term vault)

All gates share the same input: the concatenation $[h_{t-1}, x_t]$.

---

### 🔴 Gate 1 — Forget Gate ($f_t$): The Eraser
$$f_t = \sigma(W_f \cdot [h_{t-1},\, x_t] + b_f)$$

- **Output range:** $(0, 1)$ — sigmoid squashes to this range  
- **Meaning:** 0 = completely forget, 1 = completely remember  
- **Applied to:** $C_{t-1}$ — multiplied pointwise: $f_t \odot C_{t-1}$

**Example:** When the model reads a `.` (end of sentence), the forget gate fires near 0 to erase the current sentence's subject/gender, preparing for the next sentence.

---

### 🟢 Gate 2 — Input Gate ($i_t$): The Highlighter
$$i_t = \sigma(W_i \cdot [h_{t-1},\, x_t] + b_i)$$

Controls **how much** of the new candidate memory to write.

### 🟡 Candidate Memory ($\tilde{C}_t$): The Writer
$$\tilde{C}_t = \tanh(W_c \cdot [h_{t-1},\, x_t] + b_c)$$

Proposes **what** new information to add. Range $(-1, 1)$.

**Together:**
$$\text{new info added} = i_t \odot \tilde{C}_t$$

**Example:** On reading "girl", the input gate fires high and the candidate encodes "female gender = +value" into the cell state.

---

### 🔵 Cell State Update: The Memory Highway (CEC)
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

This additive update is the **Constant Error Carousel (CEC)**.  
If $f_t = 1$ and $i_t = 0$: cell state is *perfectly preserved* — the gender fact flows unchanged.

---

### 🟣 Gate 3 — Output Gate ($o_t$): The Filter
$$o_t = \sigma(W_o \cdot [h_{t-1},\, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(C_t)$$

Controls what to *reveal* from the memory vault as the visible output state.

**Example:** On reaching "her", the output gate fires to reveal the stored gender fact → model selects the correct pronoun.


In [ ]:
import numpy as np
np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

# ── Tiny LSTM: input_dim=2, hidden_dim=3 ──
D, H = 2, 3

def rand(r, c): return np.random.randn(r, c) * 0.3
def sigmoid(z): return 1 / (1 + np.exp(-z))

# Weight matrices (concatenated input [h,x] → gate)
Wf = rand(H, H+D);  bf = np.zeros(H)
Wi = rand(H, H+D);  bi = np.zeros(H)
Wc = rand(H, H+D);  bc = np.zeros(H)
Wo = rand(H, H+D);  bo = np.zeros(H)

# Sequence: 3 time steps
inputs = [np.array([0.8, 0.2]),   # step 1
          np.array([0.3, 0.9]),   # step 2
          np.array([0.1, 0.6])]   # step 3

h = np.zeros(H)   # h_0
c = np.zeros(H)   # C_0

print("=" * 65)
print("STEP-BY-STEP LSTM FORWARD PASS  (hidden_dim=3, input_dim=2)")
print("=" * 65)

for t, x in enumerate(inputs, 1):
    combined = np.concatenate([h, x])   # [h_{t-1}, x_t]

    f_t     = sigmoid(Wf @ combined + bf)
    i_t     = sigmoid(Wi @ combined + bi)
    c_tilde = np.tanh(Wc @ combined + bc)
    o_t     = sigmoid(Wo @ combined + bo)

    c_new   = f_t * c + i_t * c_tilde
    h_new   = o_t * np.tanh(c_new)

    print(f"\n── Step {t} ─────────────────────────────────────────")
    print(f"  x_{t}           = {x}")
    print(f"  h_{t-1}         = {h.round(4)}")
    print(f"  C_{t-1}         = {c.round(4)}")
    print(f"  forget gate f_{t}= {f_t.round(4)}  ← what to erase from C")
    print(f"  input gate i_{t} = {i_t.round(4)}  ← what to write")
    print(f"  candidate C̃_{t}  = {c_tilde.round(4)} ← new info content")
    print(f"  output gate o_{t} = {o_t.round(4)} ← what to reveal")
    print(f"  C_{t} = f⊙C_prev + i⊙C̃ = {c_new.round(4)}")
    print(f"  h_{t} = o⊙tanh(C_{t})  = {h_new.round(4)}")

    h, c = h_new, c_new

print("\n" + "=" * 65)
print("FINAL STATES")
print(f"  Cell State  C_3 = {c.round(4)}  ← long-term vault")
print(f"  Hidden State h_3= {h.round(4)}  ← working context")
print("=" * 65)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0f0f1a')

# ── Left: LSTM Cell Diagram ──
ax = axes[0]
ax.set_facecolor('#0f0f1a')
ax.axis('off')
ax.set_title("LSTM Cell — Data Flow", color='white', fontsize=13)

gate_data = [
    (0.18, 0.68, 'Forget\nGate f_t', '#ff6b6b',   '← eraser'),
    (0.42, 0.68, 'Input\nGate i_t',  '#fdcb6e',   '← selector'),
    (0.42, 0.38, 'Candidate\nC̃_t',   '#55efc4',   '← writer'),
    (0.66, 0.68, 'Output\nGate o_t', '#a29bfe',   '← filter'),
]

for x, y, lbl, col, sub in gate_data:
    ax.add_patch(plt.FancyBboxPatch((x-0.1, y-0.07), 0.2, 0.14,
                                    boxstyle="round,pad=0.01",
                                    facecolor=col, alpha=0.85, zorder=3))
    ax.text(x, y+0.01, lbl, ha='center', va='center', fontsize=9,
            color='black', fontweight='bold', zorder=4)
    ax.text(x, y-0.09, sub, ha='center', fontsize=8, color=col)

# Cell state highway
ax.annotate("", xy=(0.95, 0.85), xytext=(0.05, 0.85),
            arrowprops=dict(arrowstyle="->,head_width=0.3", color='#55efc4', lw=3))
ax.text(0.5, 0.9, 'Cell State Highway  C_t  (long-term memory)',
        color='#55efc4', ha='center', fontsize=10, fontweight='bold')

# Hidden state
ax.annotate("", xy=(0.95, 0.15), xytext=(0.05, 0.15),
            arrowprops=dict(arrowstyle="->,head_width=0.2", color='#fd79a8', lw=2))
ax.text(0.5, 0.08, 'Hidden State h_t  (short-term output)',
        color='#fd79a8', ha='center', fontsize=10)

# Connections from forget → cell state
for gx in [0.18, 0.42, 0.66]:
    ax.annotate("", xy=(gx, 0.85), xytext=(gx, 0.75),
                arrowprops=dict(arrowstyle="->", color='white', lw=1.5))

ax.text(0.5, 0.5, '⊕', ha='center', va='center', fontsize=28,
        color='white', fontweight='bold')
ax.text(0.5, 0.43, 'Additive\nUpdate', ha='center', va='center',
        fontsize=8, color='#aaa')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# ── Right: Gate Activations over Sequence ──
ax = axes[1]
ax.set_facecolor('#13131f')
np.random.seed(5)
T = 12
steps = np.arange(T)

# Simulate gate activations for a sentence
# "The girl who wore a beautiful dress picked up her book"
f = np.array([0.9, 0.85, 0.88, 0.9, 0.92, 0.88, 0.85, 0.9, 0.3, 0.88, 0.92, 0.9])
i = np.array([0.8, 0.6,  0.3,  0.2, 0.2,  0.25, 0.2,  0.3, 0.85,0.35, 0.2,  0.5])
o = np.array([0.5, 0.4,  0.4,  0.4, 0.4,  0.35, 0.4,  0.5, 0.4, 0.92, 0.6,  0.5])
words = ['The','girl','who','wore','a','beautiful','dress','picked','up','her','book','.']

ax.plot(steps, f, 'o-', color='#ff6b6b', lw=2, label='Forget gate ($f_t$) — what to keep')
ax.plot(steps, i, 's-', color='#fdcb6e', lw=2, label='Input gate ($i_t$)  — what to write')
ax.plot(steps, o, '^-', color='#a29bfe', lw=2, label='Output gate ($o_t$) — what to reveal')

for k, w in enumerate(words):
    ax.text(k, -0.08, w, ha='center', fontsize=7.5, color='#ccc', rotation=30)

ax.axvline(1,  color='#fdcb6e', alpha=0.3, lw=10, label='"girl" — input fires')
ax.axvline(9,  color='#a29bfe', alpha=0.3, lw=10, label='"her"  — output fires')
ax.axvline(8,  color='#ff6b6b', alpha=0.25, lw=10)

ax.set_title('Gate Activations for: "The girl … picked up her book"',
             color='white', fontsize=11)
ax.set_ylabel('Gate activation (0=off, 1=on)', color='#aaa')
ax.set_ylim(-0.25, 1.15)
ax.legend(fontsize=8, loc='upper right')
ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_color('#333')

plt.tight_layout()
plt.savefig('notes/assets/lstm_gates.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Saved → notes/assets/lstm_gates.png")


## 4. 📐 How LSTM Solves Vanishing Gradients — Mathematical Proof

### Vanilla RNN Gradient (the problem)
$$\frac{\partial C_T^{\text{RNN}}}{\partial C_k} = \prod_{i=k+1}^{T} W_{hh} \cdot \tanh'(z_i) \quad \to 0 \text{ as } T-k \to \infty$$

Each step multiplies by $W_{hh} \cdot \tanh'(z_i)$, and since both terms can be $< 1$, the product shrinks exponentially.

### LSTM Cell State Gradient
In LSTM, the cell state update is:
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

The partial derivative of $C_t$ with respect to $C_{t-1}$:
$$\frac{\partial C_t}{\partial C_{t-1}} = \text{diag}(f_t)$$

This is simply the **forget gate values** — not a product of weight matrices!

Therefore, the gradient of $C_T$ with respect to $C_k$:
$$\frac{\partial C_T}{\partial C_k} = \prod_{i=k+1}^{T} \text{diag}(f_i)$$

### Why This Doesn't Vanish

If the network learns to set $f_t \approx 1$ for a stretch of steps (i.e., "remember this"):
$$\prod_{i=k+1}^{T} f_i \approx 1^{T-k} = 1$$

The gradient is **1** — it doesn't shrink!

The network **controls its own gradient flow** via the forget gate.  
It learns: "while carrying the gender fact, keep $f_t$ near 1 so the gradient flows back cleanly."

### The Constant Error Carousel (CEC)
The term was coined by Hochreiter & Schmidhuber (1997).  
The additive cell state update creates a gradient **"carousel"** — a loop where the error signal can spin indefinitely without shrinking.

**Compare:**
| Architecture | Gradient path | Vanishes? |
|---|---|---|
| Vanilla RNN | $\prod W_{hh} \cdot \sigma'$ | ✅ Yes — exponentially |
| LSTM | $\prod f_t$ (gate values) | ❌ No — it's learnable! |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle("LSTM vs RNN — Gradient Flow Back Through Time", color='white', fontsize=13)

steps_back = np.arange(0, 60)

# RNN: W=0.9, tanh'=0.7 average
rnn_grad = (0.9 * 0.7) ** steps_back

# LSTM: forget gate ≈ 0.95 on average (learnable near 1)
lstm_grad_good = (0.95) ** steps_back
# LSTM: forget gate ≈ 0.5 (not trained yet)
lstm_grad_bad  = (0.5)  ** steps_back

ax = axes[0]
ax.set_facecolor('#13131f')
ax.semilogy(steps_back, rnn_grad,       color='#ff6b6b', lw=2.5, label='Vanilla RNN (W·tanh′=0.63)')
ax.semilogy(steps_back, lstm_grad_bad,  color='#fdcb6e', lw=2,   label='LSTM (f_t=0.5, early training)', ls='--')
ax.semilogy(steps_back, lstm_grad_good, color='#55efc4', lw=2.5, label='LSTM (f_t=0.95, trained)')
ax.axhline(1e-3, color='white', ls=':', lw=1, label='≈ learning threshold 0.001')
ax.set_xlabel('Steps back in time', color='#aaa')
ax.set_ylabel('Gradient magnitude (log scale)', color='#aaa')
ax.set_title('Gradient Magnitude vs Sequence Distance', color='white')
ax.legend(fontsize=9)
ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_color('#333')

# Cell state preservation over 50 steps
ax = axes[1]
ax.set_facecolor('#13131f')
T = 50
cell_rnn  = np.zeros(T)
cell_lstm = np.zeros(T)
val = 1.0
for t in range(T):
    val *= (0.63)
    cell_rnn[t] = val

val = 1.0
for t in range(T):
    val *= 0.95  # forget gate keeps 95% each step
    cell_lstm[t] = val

ax.plot(range(T), cell_rnn,  color='#ff6b6b', lw=2.5, label='RNN: signal after t steps')
ax.plot(range(T), cell_lstm, color='#55efc4', lw=2.5, label='LSTM cell state after t steps')
ax.fill_between(range(T), cell_rnn,  alpha=0.1, color='#ff6b6b')
ax.fill_between(range(T), cell_lstm, alpha=0.1, color='#55efc4')
ax.set_xlabel('Steps forward (sequence length)', color='#aaa')
ax.set_ylabel('Signal preservation', color='#aaa')
ax.set_title('Memory Retention: Cell State vs RNN Hidden State', color='white')
ax.legend(fontsize=10)
ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_color('#333')

plt.tight_layout()
plt.savefig('notes/assets/lstm_gradient.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


## 5. 🏗️ LSTM Variants

### 5.1 Stacked (Deep) LSTM
Multiple LSTM layers stacked vertically. The hidden state $h_t^{(l)}$ of layer $l$ is the input to layer $l+1$.
- **More layers** = more abstraction capacity  
- Common: 2–4 layers for NLP tasks  
- Add `dropout` between layers to prevent overfitting

### 5.2 Bidirectional LSTM
Same as bidirectional RNN — two LSTM layers (forward + backward).  
Final representation per step: $H_t = [\overrightarrow{h}_t ; \overleftarrow{h}_t]$  
Used in BERT, NER, POS tagging.

### 5.3 Peephole LSTM
Standard gates use only $[h_{t-1}, x_t]$ as input.  
Peephole connections also feed $C_{t-1}$ into all gates:
$$f_t = \sigma(W_f \cdot [C_{t-1}, h_{t-1}, x_t] + b_f)$$

**When to use:** Tasks requiring precise temporal event detection — e.g., speech onset detection, music beat tracking. The gates need to know *exactly* how full the memory vault is, not just what the last output was.


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

# ── Manual LSTM Cell (transparent, shows all 4 gates) ──
class ManualLSTMCell(nn.Module):
    """Transparent LSTM cell exposing all 4 gate activations."""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # One fused weight matrix for speed: 4 gates × hidden_dim
        self.W = nn.Linear(input_dim + hidden_dim, 4 * hidden_dim, bias=True)
        # Bias trick: init forget gate bias to 1.0 (avoid initial amnesia)
        nn.init.constant_(self.W.bias[hidden_dim:2*hidden_dim], 1.0)

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([h_prev, x], dim=1)
        gates    = self.W(combined)
        H = gates.shape[1] // 4
        f_raw, i_raw, o_raw, c_raw = gates.split(H, dim=1)

        f = torch.sigmoid(f_raw)        # forget gate
        i = torch.sigmoid(i_raw)        # input gate
        o = torch.sigmoid(o_raw)        # output gate
        c_tilde = torch.tanh(c_raw)     # candidate

        c_next = f * c_prev + i * c_tilde   # additive CEC update
        h_next = o * torch.tanh(c_next)

        return h_next, c_next, {'f': f, 'i': i, 'o': o, 'c_tilde': c_tilde}

# ── Stacked BiLSTM Classifier ──
class StackedBiLSTM(nn.Module):
    """2-layer Bidirectional LSTM for text classification."""
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2,
                 num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for BiLSTM

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        out, (h, c) = self.lstm(emb)
        # Concatenate final forward and backward layer
        ctx = torch.cat([h[-2], h[-1]], dim=1)   # (batch, hidden*2)
        return self.fc(self.dropout(ctx))

# ── Instantiate and inspect ──
model = StackedBiLSTM(vocab_size=10000)
x = torch.randint(1, 10000, (4, 30))  # batch=4, seq=30
out = model(x)
print("Stacked BiLSTM")
print(f"  Input : {x.shape}")
print(f"  Output: {out.shape}  (4 samples, 2-class logits)")
total = sum(p.numel() for p in model.parameters())
print(f"  Params: {total:,}")

# ── Show gate activation from ManualCell ──
cell = ManualLSTMCell(input_dim=8, hidden_dim=16)
h, c = torch.zeros(1, 16), torch.zeros(1, 16)
gate_log = {'f':[], 'i':[], 'o':[]}
for t in range(10):
    x_t = torch.randn(1, 8)
    h, c, gates = cell(x_t, h, c)
    for k in ['f','i','o']:
        gate_log[k].append(gates[k].mean().item())

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#0f0f1a')
ax.set_facecolor('#13131f')
colors = {'f':'#ff6b6b','i':'#fdcb6e','o':'#a29bfe'}
labels = {'f':'Forget gate','i':'Input gate','o':'Output gate'}
for k in ['f','i','o']:
    ax.plot(gate_log[k], 'o-', color=colors[k], lw=2, label=labels[k])
ax.set_xlabel('Time step', color='#aaa')
ax.set_ylabel('Mean gate activation', color='#aaa')
ax.set_title('ManualLSTMCell — Gate Activations over 10 Steps
(random input, random weights)', color='white')
ax.legend(); ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_color('#333')
plt.tight_layout()
plt.savefig('notes/assets/lstm_cell_gates.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


## 6. 🛠️ End-to-End Training — IMDb-style Sentiment (BiLSTM)

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt
torch.manual_seed(99)

# ── Synthetic data mimicking IMDb binary classification ──
VOCAB  = 3000
SEQ    = 25
BATCH  = 16

def make_data(n, pos=True):
    X = torch.randint(1, VOCAB, (n, SEQ))
    if pos:
        # "Good" words cluster in mid vocab range
        X[:, :5] = torch.randint(1500, 2000, (n, 5))
    else:
        X[:, :5] = torch.randint(200, 700, (n, 5))
    y = torch.ones(n, dtype=torch.long) if pos else torch.zeros(n, dtype=torch.long)
    return X, y

Xp, yp = make_data(200, True)
Xn, yn = make_data(200, False)
X_all  = torch.cat([Xp, Xn])
y_all  = torch.cat([yp, yn])
# Shuffle
perm   = torch.randperm(400)
X_all, y_all = X_all[perm], y_all[perm]

X_tr, y_tr = X_all[:300], y_all[:300]
X_va, y_va = X_all[300:], y_all[300:]

class SentimentLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(VOCAB, 32, padding_idx=0)
        self.lstm = nn.LSTM(32, 64, num_layers=2, batch_first=True,
                            bidirectional=True, dropout=0.3)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(128, 2)
    def forward(self, x):
        e = self.drop(self.emb(x))
        _, (h, _) = self.lstm(e)
        ctx = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.drop(ctx))

model  = SentimentLSTM()
opt    = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
loss_fn= nn.CrossEntropyLoss()

train_losses, val_accs = [], []

for epoch in range(60):
    model.train()
    perm = torch.randperm(300)
    ep_loss = 0
    for b in range(0, 300, BATCH):
        idx  = perm[b:b+BATCH]
        xb, yb = X_tr[idx], y_tr[idx]
        logits = model(xb)
        loss   = loss_fn(logits, yb)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        ep_loss += loss.item()
    sched.step()
    train_losses.append(ep_loss / (300 // BATCH))

    model.eval()
    with torch.no_grad():
        val_pred = model(X_va).argmax(1)
        val_acc  = (val_pred == y_va).float().mean().item()
    val_accs.append(val_acc)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes: ax.set_facecolor('#13131f')

axes[0].plot(train_losses, color='#4ecdc4', lw=2, label='Train loss')
axes[0].fill_between(range(len(train_losses)), train_losses, alpha=0.15, color='#4ecdc4')
axes[0].set_title('Training Loss (CrossEntropy)', color='white')
axes[0].set_xlabel('Epoch', color='#aaa'); axes[0].legend()
axes[0].tick_params(colors='#aaa')

axes[1].plot(val_accs, color='#fd79a8', lw=2, label='Val accuracy')
axes[1].axhline(0.5, color='#fdcb6e', ls='--', lw=1, label='Random baseline')
axes[1].fill_between(range(len(val_accs)), val_accs, alpha=0.15, color='#fd79a8')
axes[1].set_title(f'Validation Accuracy (Final: {val_accs[-1]:.1%})', color='white')
axes[1].set_xlabel('Epoch', color='#aaa'); axes[1].legend()
axes[1].tick_params(colors='#aaa')

for ax in axes:
    for sp in ax.spines.values(): sp.set_color('#333')

plt.suptitle("Stacked BiLSTM — Sentiment Training", color='white', fontsize=13)
plt.tight_layout()
plt.savefig('notes/assets/lstm_training.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print(f"Final val accuracy: {val_accs[-1]:.1%}")


## 7. 📋 LSTM Reference — Summary Tables

### Gate Summary
| Gate | Formula | Range | Role | Analogy |
|---|---|---|---|---|
| Forget    | $\sigma(W_f[h,x]+b_f)$ | (0,1) | Erase from cell state | Whiteboard eraser |
| Input     | $\sigma(W_i[h,x]+b_i)$ | (0,1) | Control write amount | Pen valve |
| Candidate | $\tanh(W_c[h,x]+b_c)$ | (-1,1)| What content to write | Ink colour |
| Output    | $\sigma(W_o[h,x]+b_o)$ | (0,1) | Filter what to reveal | Curtain |

### LSTM vs RNN
| Dimension | RNN | LSTM |
|---|---|---|
| States | 1 ($h_t$) | 2 ($h_t$, $C_t$) |
| Gradient path | Multiplicative → vanishes | Additive CEC → stable |
| Parameters (H=128) | ~33K | ~264K (4× gates) |
| Long-range deps | ❌ | ✅ |
| Training speed | Faster | Slower (4× compute) |

---

## 8. 🎯 Senior-Level Interview Q&A

**Q1: Why does cell state use addition while hidden state uses multiplication?**  
> The cell state uses ADDITION ($f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$) so the gradient can flow back without shrinking. Hidden state uses element-wise multiplication with tanh output to produce a bounded, useful hidden representation. Mixing addition (for memory) and multiplication (for gating) is the key design insight.

---

**Q2: Why initialise forget gate bias to 1.0?**  
> At the start of training, we want the LSTM to remember everything by default. Setting $b_f = 1$ makes $f_t = \sigma(1) \approx 0.73$ right away — the model starts "mostly remembering" and then learns what to forget. Starting with $b_f = 0$ means $f_t = 0.5$, which discards half the memory from step 1 — the model never learns long-range dependencies in early epochs.

---

**Q3: When would LSTM fail and you'd need a Transformer?**  
> When sequences are very long (>500 tokens), even LSTM's Cell State becomes hard to optimise. Also, LSTM cannot parallelise across time — training on 10K-long sequences is very slow. Transformers use attention to directly attend to any position in O(1) steps, making them superior for very long sequences and large-scale parallel training.

---

**Q4: What is the "Constant Error Carousel" formally?**  
> It's the name for the gradient path through the cell state's additive update. Because $\partial C_t / \partial C_{t-1} = \text{diag}(f_t)$, which is NOT a product of the weight matrix, the gradient from step $T$ to step $k$ equals $\prod f_i$ — a product of gate values, not weights. The network can learn $f_i \approx 1$ to make this product $\approx 1$, keeping the gradient alive. The "carousel" metaphor: the error signal keeps spinning without fading.

---

**Q5: Can LSTM forget MORE than it remembers from the beginning?**  
> Yes — if the forget gate fires near 0 consistently (e.g., after reading a sentence-ending period), the cell state is nearly zeroed. This is intentional in multi-sentence tasks like document classification, where the model must "reset" between sentences while still carrying some document-level context via the hidden state.
